# Python Fundamentals for AI Engineering
### Explained plainly -- what each thing is, when to use it, when NOT to, and the real trade-off

Every topic below follows the same shape:
- **What it is** -- in plain words, no jargon
- **A simple example** -- generic, everyday scenarios, nothing tied to any specific project
- **Where you'd actually use it** vs **where you shouldn't**
- **Pros and cons**

## 1. `*args` and `**kwargs`

**What it is:** A normal function only accepts exactly the arguments you define:

```python
def add(a, b):
    return a + b
```

This only works with exactly 2 arguments. `*args` and `**kwargs` let a function accept an *unknown* number of arguments instead -- Python collects the extra ones for you rather than erroring out.

- `*args` -- collects extra arguments passed **without** a name, into a tuple
- `**kwargs` -- collects extra arguments passed **with** a name (like `size="large"`), into a dictionary

These are just variable names by convention -- you could call them anything, but everyone recognizes `args`/`kwargs` specifically, so stick with that.

In [1]:
def show_everything(*args, **kwargs):
    print("args:", args)
    print("kwargs:", kwargs)

show_everything(1, 2, "hello", name="Alex", age=30)
# args: (1, 2, 'hello')
# kwargs: {'name': 'Alex', 'age': 30}

args: (1, 2, 'hello')
kwargs: {'name': 'Alex', 'age': 30}


**Where you'd use it:** Only when you're writing something *generic* that has to work with functions you don't know in advance -- the main real case is a wrapper function, like something that logs or times ANY other function, no matter what arguments that function needs.

In [2]:
def log_before_calling(func, *args, **kwargs):
    "This can wrap ANY function -- it doesn't need to know its arguments ahead of time."
    print(f"About to call {func.__name__} with args={args}, kwargs={kwargs}")
    return func(*args, **kwargs)

def multiply(a, b):
    return a * b

def greet(name, greeting="Hello"):
    return f"{greeting}, {name}!"

print(log_before_calling(multiply, 3, 4))
print(log_before_calling(greet, "Sam", greeting="Hi"))

About to call multiply with args=(3, 4), kwargs={}
12
About to call greet with args=('Sam',), kwargs={'greeting': 'Hi'}
Hi, Sam!


**Where you should NOT use it:** If a function always takes a known, fixed set of inputs -- just write out the named parameters:

```python
def calculate_total(price, quantity, discount=0):
    ...
```

**Pros:**
- Lets one function wrap/forward to *any* other function, regardless of its signature

**Cons:**
- You lose clarity -- someone reading `log_before_calling(func, *args, **kwargs)` can't tell what's actually expected without also reading `func` itself
- You lose IDE/type-checking help -- your editor can't warn you "you forgot an argument" when everything's swallowed into `*args`/`**kwargs`

**Rule of thumb:** default to named parameters. Only reach for `*args`/`**kwargs` when building a generic wrapper around functions you don't control.

## 2. List and Dict Comprehensions

**What it is:** A compact way to build a new list (or dict, or set) from an existing one, in a single line, instead of writing a manual loop that appends to an empty list.

The old-style loop way:
```python
result = []
for x in some_list:
    result.append(x.upper())
```

The comprehension way does the exact same thing in one line.

In [3]:
fruits = ["apple", "banana", "cherry", "date"]

# list comprehension -- same result as the manual loop above
uppercased = [f.upper() for f in fruits]
print(uppercased)

# you can add a condition -- only include items that pass a check
long_names = [f for f in fruits if len(f) > 5]
print(long_names)

# dict comprehension -- same idea, but building a dictionary
name_lengths = {f: len(f) for f in fruits}
print(name_lengths)

['APPLE', 'BANANA', 'CHERRY', 'DATE']
['banana', 'cherry']
{'apple': 5, 'banana': 6, 'cherry': 6, 'date': 4}


**Where you'd use it:** Any time you're transforming or filtering a list/dict in one pass -- this is genuinely the default, idiomatic way to do it in Python, not an advanced trick.

**Where you should NOT use it:** If the logic inside is more than one simple expression (multiple `if` conditions, or a multi-step calculation), a comprehension becomes harder to read than a plain loop. Don't force it just to be "Pythonic" -- readability wins.

**Pros:**
- Shorter, and once you're used to reading them, faster to understand than a 3-line loop
- Slightly faster to run than the manual loop version (minor, rarely the deciding factor)

**Cons:**
- A comprehension crammed with multiple conditions gets genuinely hard to read -- worse than the loop it replaced

## 3. f-strings (String Formatting)

**What it is:** The modern way to build a string that includes variable values. You put an `f` right before the quote, and anything inside `{ }` gets evaluated and inserted.

In [4]:
name = "Priya"
age = 30

# f-string -- put the f before the quotes, use {} for any Python expression
message = f"{name} is {age} years old"
print(message)

# you can even do calculations or formatting inside the braces
price = 19.999
print(f"Price: ${price:.2f}")  # :.2f means "2 decimal places"

Priya is 30 years old
Price: $20.00


**Where you'd use it:** Basically always, whenever you're building a string that includes a variable. It's the modern default in any Python code written in the last several years.

**Where you should NOT use it:** Very long, multi-line text templates (like an email body) are sometimes clearer built with `.format()` or a template library -- but for normal short messages, f-strings are always the right choice.

**Pros:**
- Easy to read -- the variable is right there in the string, not off to the side

**Cons:**
- None really -- just don't forget the `f` before the quote, or `{name}` prints literally instead of substituting the value

## 4. Unpacking, `enumerate`, and `zip`

**What it is:** Three small tools for avoiding manual index bookkeeping (`list[0]`, `list[1]`, tracking a counter yourself) when looping.

- **Unpacking** -- pulling multiple values out of a tuple/list directly into named variables
- **`enumerate`** -- gives you both the index AND the item while looping, without you tracking a counter yourself
- **`zip`** -- walks two (or more) lists together, pairing up matching positions

In [5]:
students = [("Maya", 92), ("Leo", 78), ("Sara", 85)]

# unpacking directly in the loop -- no students[i][0] / students[i][1] indexing
for name, score in students:
    print(f"{name}: {score}")

print()

# enumerate -- when you DO need the position too
for i, (name, score) in enumerate(students, start=1):
    print(f"#{i} {name}: {score}")

print()

# zip -- walking two separate lists together
names = ["Maya", "Leo"]
scores = [92, 78]
for name, score in zip(names, scores):
    print(f"{name} scored {score}")

Maya: 92
Leo: 78
Sara: 85

#1 Maya: 92
#2 Leo: 78
#3 Sara: 85

Maya scored 92
Leo scored 78


**Where you'd use it:** Any loop where you'd otherwise be writing `list[i]` or manually incrementing a counter. These three cover the vast majority of those cases cleanly.

**Where you should NOT use it:** If you genuinely need to modify a list while looping over it by index, plain indexing is sometimes clearer -- but this is a rare case.

**Pros:**
- Removes an entire category of bugs (off-by-one errors from manual index tracking)

**Cons:**
- None significant -- these are strictly safer/cleaner than manual indexing in almost every case

## 5. Generators (`yield`)

**What it is:** A normal function runs top to bottom and returns one result. A **generator** function uses `yield` instead of `return`, and produces values **one at a time, on demand**, instead of building the entire result upfront and holding it all in memory at once.

In [6]:
def count_up_to(n):
    "Produces numbers ONE AT A TIME instead of building a full list upfront."
    current = 1
    while current <= n:
        yield current
        current += 1

# Calling this does NOT actually produce any numbers yet -- it just creates
# a generator object, ready to produce values when asked
gen = count_up_to(1_000_000)
print(type(gen))

# each loop step computes exactly ONE number, not all a million at once
for i, number in enumerate(gen):
    if i >= 5:
        break
    print(number)

<class 'generator'>
1
2
3
4
5


**Where you'd use it:** Processing anything large -- reading a huge file line by line, working through a big dataset, streaming results as they arrive. The key benefit is memory: you never hold the *entire* result set in memory at once, just whatever you're currently working on.

**Where you should NOT use it:** Small, fixed-size data where you need the *entire* result immediately anyway (e.g. you need to sort it, or check its length) -- at that point a plain list is simpler and there's no memory benefit to lose.

**Pros:**
- Handles data too large to fit in memory all at once
- Doesn't do any work until you actually ask for the next value ("lazy evaluation")

**Cons:**
- You can only loop through a generator ONCE -- unlike a list, you can't go back and loop over it again without recreating it
- Harder to debug -- you can't just print a generator and see all its values, since they don't exist yet until asked for

## 6. Decorators

**What it is:** A decorator is a function that **wraps another function** to add behavior around it, without changing the original function's code. The `@` symbol is just shorthand for "wrap this function with that other function."

### First -- the exact order things run in, with print statements at every step
This is the part that's genuinely confusing at first, so let's trace it line by line.

In [7]:
def my_decorator(func):
    print("1. Decorator is running -- wrapping the function now")

    def wrapper():
        print("3. Wrapper starts (this runs when you actually CALL the function)")
        func()
        print("5. Wrapper ends")

    print("2. Decorator finished wrapping -- returning the wrapper")
    return wrapper

@my_decorator
def say_hello():
    print("4. Inside the actual original function")

print("--- now calling say_hello() ---")
say_hello()

1. Decorator is running -- wrapping the function now
2. Decorator finished wrapping -- returning the wrapper
--- now calling say_hello() ---
3. Wrapper starts (this runs when you actually CALL the function)
4. Inside the actual original function
5. Wrapper ends


**The key insight -- there are TWO separate moments, not one:**

**Moment 1 -- when Python first reads the file** (the `@my_decorator` line), the decorator function runs **immediately**, right then -- that's steps 1 and 2, happening *before* you've even called `say_hello()` yet. All the decorator does at this moment is build the `wrapper` function and hand it back.

**Moment 2 -- later, when you actually call `say_hello()`**, you're not calling the original function directly anymore. `@my_decorator` secretly replaced `say_hello` with `wrapper`. So calling `say_hello()` actually runs `wrapper()` -- that's steps 3, 4, 5 -- and *inside* `wrapper` is where your original function finally gets called, sandwiched in the middle.

So: the decorator itself runs once, immediately, at definition time. Then every time you actually *call* the decorated function afterward, it's the wrapper that runs, and the wrapper decides when (and whether) to call your original function.

### A more realistic use case -- timing how long a function takes

In [8]:
import time
import functools

def timed(func):
    @functools.wraps(func)  # keeps the original function's name/docstring intact -- always include this
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        print(f"{func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper

@timed
def slow_addition(a, b):
    time.sleep(0.1)  # pretending this takes a while
    return a + b

result = slow_addition(3, 4)
print("Result:", result)

# The @timed line above is EXACTLY equivalent to writing:
#   slow_addition = timed(slow_addition)

slow_addition took 0.1001s
Result: 7


**Where you'd use it:** Adding the SAME behavior (timing, logging, retrying, checking permissions) around many different functions, without copy-pasting that logic into every one of them.

**Where you should NOT use it:** If you only need this behavior in ONE place, just write it inline -- a decorator adds a layer of indirection that isn't worth it for a single use.

**Pros:**
- Write the extra behavior once, apply it anywhere with one line (`@timed`)
- Keeps the original function's code completely clean and focused on its actual job

**Cons:**
- Adds a layer of indirection -- when reading `slow_addition(...)`, you have to remember it's not *really* running `slow_addition` directly, it's running through `wrapper` first

## 7. Context Managers (`with` / `as`)

**What it is:** A way to guarantee some cleanup code runs, **even if something goes wrong** in between. You've already used this every time you write `with open(file) as f:`.

The guarantee is the whole point: whatever's inside the `with` block, the cleanup at the end always runs -- whether the block finished normally or raised an exception partway through.

In [9]:
class Timer:
    def __enter__(self):
        # runs when the `with` block STARTS
        self.start = time.time()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        # runs when the `with` block ENDS -- even if it ended because of an error
        elapsed = time.time() - self.start
        print(f"Block took {elapsed:.4f}s")
        return False  # False means: don't hide any exception, let it propagate normally

with Timer():
    time.sleep(0.05)
    print("doing work inside the block")
# Timer's __exit__ runs automatically here -- guaranteed, even if the code
# above had raised an exception instead of completing normally

doing work inside the block
Block took 0.0503s


**Where you'd use it:** Anything that needs guaranteed cleanup -- closing a file, closing a database connection, releasing a lock.

**Where you should NOT use it:** If there's genuinely nothing to clean up (no file, no connection, no resource to release), you don't need this -- a plain function call is fine.

**Pros:**
- Guarantees cleanup happens, even on error -- this is very hard to get right manually with try/finally scattered everywhere
- Very readable -- `with X() as x:` clearly signals "this resource is scoped to this block"

**Cons:**
- Requires understanding `__enter__`/`__exit__` if you're writing your OWN context manager (using an existing one, like `open()`, requires none of that)

## 8. Type Hints

**What it is:** Optional notes in your function signature saying what type each argument and the return value should be. Python does **not** enforce these at runtime -- they're purely for humans (and your editor) to read, not a hard rule Python checks.

In [10]:
def calculate_total(price: float, quantity: int) -> float:
    "The hints here: price should be a float, quantity an int, return is a float."
    return price * quantity

print(calculate_total(9.99, 3))

29.97


In [11]:
# Python does NOT stop you from calling this with the WRONG type -- the
# type hint is purely documentation, not an enforced rule. Watch what
# happens: it does not fail immediately at the call, it fails LATER,
# in a way that may not obviously point back to "you passed the wrong type":
try:
    result = calculate_total(9.99, "three")  # quantity should be an int, not a string
except TypeError as e:
    print(f"Failed, but NOT at the call site -- inside the function: {e}")
    print("This is exactly why type hints matter: they would have let your")
    print("editor warn you about this BEFORE running, instead of finding out")
    print("via a runtime error.")


Failed, but NOT at the call site -- inside the function: can't multiply sequence by non-int of type 'float'
This is exactly why type hints matter: they would have let your
editor warn you about this BEFORE running, instead of finding out
via a runtime error.


**Where you'd use it:** Every function you write, as a habit. It costs almost nothing and makes your code self-documenting -- you know immediately what goes in and what comes out, without reading the function body.

**Where you should NOT use it:** There's essentially no case where type hints hurt -- the only real exception is quick throwaway scratch scripts where the extra typing isn't worth your time.

**Pros:**
- Makes function signatures self-documenting
- Your editor/IDE uses them to warn you about mistakes before you even run the code

**Cons:**
- Not enforced at runtime -- a type hint doesn't actually stop wrong data from being passed in, it's a documentation/tooling aid, not a safety net by itself

## 9. Exception Handling and Custom Exceptions

**What it is:** `try`/`except` lets you catch an error instead of letting it crash your program, and respond to it deliberately. A **custom exception** is your own named error type (instead of always using Python's generic built-in ones) -- useful when you want the error itself to clearly say WHAT went wrong.

**The most important lesson here:** a caught error should never be silently turned into something that looks like a normal, successful result.

In [12]:
class InsufficientFundsError(Exception):
    "A custom exception -- just a named class that inherits from Exception."
    pass

def withdraw(balance: float, amount: float) -> float:
    if amount > balance:
        raise InsufficientFundsError(
            f"Cannot withdraw {amount}, balance is only {balance}"
        )
    return balance - amount

try:
    withdraw(balance=100, amount=150)
except InsufficientFundsError as e:
    print(f"Caught explicit failure: {e}")

Caught explicit failure: Cannot withdraw 150, balance is only 100


**Why this matters more than it might seem:** imagine `withdraw` had instead just returned `0` or `None` when there wasn't enough money, instead of raising an error. To whoever calls it, that could look exactly like "the withdrawal succeeded and the new balance is 0" -- a silent, dangerous mix-up between "failed" and "succeeded with an unusual result." Raising a clear, named exception avoids that ambiguity entirely.

**Where you'd use it:** Any time a function might fail in a way the caller needs to know about and react to differently than "everything's fine." Custom exceptions are worth creating whenever generic Python errors (`ValueError`, `Exception`) wouldn't clearly say WHAT went wrong in your specific situation.

**Where you should NOT use it:** Don't wrap `try`/`except` around code that genuinely can't fail, just out of caution -- it adds noise and can accidentally hide real bugs if the `except` is too broad (catching `Exception` generically instead of the specific error you expect).

**Pros:**
- Prevents the single most dangerous failure mode: a broken call that silently looks like a successful result
- Custom exception names make error logs immediately tell you WHAT kind of problem happened

**Cons:**
- Overly broad `except Exception:` blocks can accidentally swallow bugs you actually wanted to see -- always catch the most SPECIFIC error type you can

## 10. Lambda Functions

**What it is:** A tiny, unnamed function written in a single line, normally used only when you need a small function to pass into something else (like a sort key), not for anything you'd want to reuse or name.

```python
lambda x: x * 2
```
is a quick, throwaway version of:
```python
def double(x):
    return x * 2
```

In [13]:
words = ["banana", "kiwi", "watermelon", "fig"]

# lambda used as a "sort key" -- telling sort HOW to rank each item,
# without writing a full separate named function just for this one use
sorted_by_length = sorted(words, key=lambda w: len(w))
print(sorted_by_length)

# same idea, sorting numbers by how close they are to 10
numbers = [1, 15, 8, 22, 9]
sorted_by_closeness = sorted(numbers, key=lambda n: abs(n - 10))
print(sorted_by_closeness)

['fig', 'kiwi', 'banana', 'watermelon']
[9, 8, 15, 1, 22]


**Where you'd use it:** Small, throwaway logic passed directly into something like `sorted()`, `filter()`, or `map()`.

**Where you should NOT use it:** Anything more than a single simple expression. If you need more than one line of logic, or you want to reuse the same logic in multiple places, write a normal named `def` function instead -- a lambda with complex logic crammed into it is hard to read.

**Pros:**
- Very compact for simple, one-off logic passed into another function

**Cons:**
- No name, so error messages referencing it are less helpful ("error in <lambda>" instead of a real function name)
- Easy to misuse for logic that's actually complex enough to deserve a real named function

## 11. Dataclasses

**What it is:** A shortcut for writing a class that's mostly just a bundle of named fields, with no complex behavior. Normally you'd hand-write `__init__` to accept and store each field -- `@dataclass` generates that for you automatically.

In [14]:
from dataclasses import dataclass, field

@dataclass
class Book:
    title: str
    author: str
    year: int
    tags: list = field(default_factory=list)
    # IMPORTANT gotcha: never write `tags: list = []` directly -- a plain
    # mutable default like that gets SHARED across every instance of Book,
    # which is a classic, confusing Python bug. default_factory avoids it.

book = Book(title="Dune", author="Frank Herbert", year=1965)
print(book)              # @dataclass auto-generates a readable print format for free
print(book.title)        # access fields like normal attributes

Book(title='Dune', author='Frank Herbert', year=1965, tags=[])
Dune


**Where you'd use it:** Any class whose main job is holding a fixed set of named values together -- a book record, a config object, a single data point. If you're mostly writing `self.x = x` lines in `__init__` and nothing else complex, it should probably be a dataclass.

**Where you should NOT use it:** A class with real behavior -- lots of methods, validation logic, custom construction rules -- benefits less from `@dataclass`, since the auto-generated `__init__` might not be flexible enough.

**Pros:**
- No hand-written `__init__` boilerplate
- Free, readable `print()` output out of the box

**Cons:**
- The mutable-default gotcha shown above is a real, commonly-hit bug if you forget `default_factory`

## 12. Inheritance and the "Dunder" Methods That Actually Matter

**What it is:** Inheritance means one class builds on another (`class Dog(Animal):` means "Dog IS a kind of Animal, with all of Animal's behavior, plus whatever I add"). "Dunder" methods (double-underscore, like `__repr__`) are special method names Python recognizes and calls automatically in specific situations -- you rarely call them directly yourself.

Two worth knowing, because they show up constantly:
- `__repr__` -- controls what shows up when you `print()` an object
- `__call__` -- lets you use an object like a function: `my_object(...)`

In [15]:
class Animal:
    def __init__(self, name):
        self.name = name
    def speak(self):
        return "..."

class Dog(Animal):
    def speak(self):
        # overriding the parent's method with dog-specific behavior
        return "Woof!"

fido = Dog("Fido")
print(fido.name)     # inherited from Animal, unchanged
print(fido.speak())  # Dog's own version

Fido
Woof!


In [16]:
class Calculator:
    def __init__(self, name):
        self.name = name

    def __repr__(self):
        # without this, print(calc) would show something unhelpful like
        # <__main__.Calculator object at 0x7f...> -- always define this
        # on classes you'll want to inspect/debug
        return f"Calculator(name={self.name!r})"

    def __call__(self, a, b):
        # this is what makes `calc(2, 3)` work, as if calc were a function
        return a + b

calc = Calculator("Adder")
print(calc)          # uses __repr__ automatically
print(calc(2, 3))    # uses __call__ automatically -- no .run() needed

Calculator(name='Adder')
5


**Where you'd use it:** `__repr__` -- on almost every class you write, it's low-effort and saves real debugging time. `__call__` -- specifically when an object's whole purpose is "do one main action," and making it directly callable reads more naturally than `.run()` or `.execute()`.

**Where you should NOT use it:** Don't reach for `__call__` just because it seems clever -- if a class has several distinct actions it can do, separate named methods are clearer than cramming everything into one `__call__`.

**Pros:**
- `__repr__` makes debugging significantly faster -- you see meaningful output instead of a memory address
- `__call__` can make an object's usage read very naturally when it truly does one thing

**Cons:**
- Overusing dunder methods makes code harder to predict -- use them only where the special behavior genuinely matches the class's purpose

## 13. Properties

**What it is:** A way to make a method LOOK like a plain attribute from the outside (`circle.area`, no parentheses) while it's actually running code every time it's accessed. Useful for values that are *computed from* other data, so they're always automatically up to date.

In [17]:
import math

class Circle:
    def __init__(self, radius):
        self.radius = radius

    @property
    def area(self) -> float:
        # looks like a plain field from outside, but is recalculated
        # fresh every single time it's accessed -- never goes stale
        return math.pi * self.radius ** 2

c = Circle(radius=5)
print(c.area)  # no parentheses -- reads like a plain attribute

c.radius = 10  # change the radius...
print(c.area)  # ...and area is automatically correct next time you check it

78.53981633974483
314.1592653589793


**Where you'd use it:** Any value that's always derived from other data on the object, where you want callers to access it simply (`obj.area`) without needing to know it's actually computed.

**Where you should NOT use it:** If the computation is genuinely expensive (a slow database call, for instance), a property can be misleading -- callers expect attribute access to be instant, so an expensive property can surprise people with unexpected slowness. A named method (`obj.compute_area()`) signals "this might take a moment" more honestly.

**Pros:**
- Never goes out of sync, since it's recalculated every access
- Clean, attribute-style syntax for the caller

**Cons:**
- Hides the fact that real computation is happening behind what looks like simple field access

## 14. Data Structures -- Which One, and Why

**What it is:** Python's 4 main built-in containers, and when each is the right choice:

| Structure | Ordered? | Duplicates? | Fast at |
|---|---|---|---|
| `list` | Yes | Yes | Adding to the end, accessing by position |
| `tuple` | Yes | Yes | Same as list, but CANNOT be changed after creation |
| `dict` | Yes (insertion order) | Keys must be unique | Looking things up by a key -- very fast |
| `set` | No | No duplicates allowed | Checking "is this item in here?" -- very fast |

The `set` row is the one worth internalizing: checking membership in a `set` is dramatically faster than checking membership in a `list`, especially as the collection grows.

In [18]:
allowed_colors = {"red", "green", "blue", "yellow"}  # this is a set

# checking "is this color in the set" is fast regardless of how many
# colors are in there -- a list would get slower as it grows, a set does not
print("red" in allowed_colors)
print("purple" in allowed_colors)

True
False


In [19]:
from collections import defaultdict, Counter, deque

# --- defaultdict: skips the "if key not in dict: dict[key] = []" step ---
students_by_grade = defaultdict(list)
for name, grade in [("Maya", "A"), ("Leo", "B"), ("Sara", "A")]:
    students_by_grade[grade].append(name)
print(dict(students_by_grade))

# --- Counter: built specifically for tallying things up ---
votes = ["cat", "dog", "cat", "cat", "dog", "fish"]
print(Counter(votes))
print(Counter(votes).most_common(1))  # the single most common item

# --- deque: a list that's fast to add/remove from BOTH ends ---
# useful for "keep only the last N things"
recent_searches = deque(maxlen=3)  # automatically drops the oldest when full
for search in ["cats", "dogs", "birds", "fish"]:
    recent_searches.append(search)
print(list(recent_searches))  # "cats" got dropped automatically

{'A': ['Maya', 'Sara'], 'B': ['Leo']}
Counter({'cat': 3, 'dog': 2, 'fish': 1})
[('cat', 3)]
['dogs', 'birds', 'fish']


**Where you'd use each:** `list` for ordered data you'll loop through or append to. `dict` when you need to look things up by name/key. `set` whenever you're checking "have I seen this before" or "is this allowed" repeatedly. `tuple` for a small, fixed group of values that should never change (like a coordinate pair). `defaultdict`/`Counter`/`deque` are specialized shortcuts for extremely common patterns.

**Where you should NOT use it:** Don't reach for a `set` if you actually need to preserve order or allow duplicates -- it silently drops duplicates and doesn't guarantee order the way a list does.

**Pros:**
- Choosing the right structure isn't just style -- `set` membership checks are genuinely, measurably faster than `list` ones at scale
- The `collections` shortcuts remove a lot of repetitive boilerplate

**Cons:**
- `defaultdict`'s "just make an empty one" behavior can mask a real bug (a key you expected to already exist, but doesn't)

## 15. Calling APIs -- `requests` (Synchronous)

**What it is:** `requests` is the standard, simplest way to call an external API from Python -- send a request, wait (block) until the response comes back, then continue.

The example below sets up a tiny local test server first, so this runs reliably without depending on any external website being reachable.

In [20]:
import json as _json
import threading
import time as _time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import urlparse, parse_qs

class MockHandler(BaseHTTPRequestHandler):
    def log_message(self, *args):
        pass

    def do_GET(self):
        parsed = urlparse(self.path)
        if parsed.path == "/get":
            params = {k: v[0] for k, v in parse_qs(parsed.query).items()}
            body = _json.dumps({"args": params}).encode()
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(body)
        elif parsed.path.startswith("/delay/"):
            seconds = float(parsed.path.split("/delay/")[1])
            _time.sleep(seconds)
            body = _json.dumps({"delayed_seconds": seconds}).encode()
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(body)
        else:
            self.send_response(404)
            self.end_headers()

_server = ThreadingHTTPServer(("127.0.0.1", 8765), MockHandler)
_thread = threading.Thread(target=_server.serve_forever, daemon=True)
_thread.start()
MOCK_BASE_URL = "http://127.0.0.1:8765"
print(f"Local test server running at {MOCK_BASE_URL}")

Local test server running at http://127.0.0.1:8765


In [21]:
import requests

response = requests.get(f"{MOCK_BASE_URL}/get", params={"search": "python books"})
print(response.status_code)       # 200 means success
print(response.json())            # parse the response body as JSON

# .raise_for_status() turns a failed response (4xx/5xx) into a Python
# exception, instead of you having to manually check status_code every time
try:
    response.raise_for_status()
    print("Request succeeded")
except requests.HTTPError as e:
    print(f"Request failed: {e}")

200
{'args': {'search': 'python books'}}
Request succeeded


**Where you'd use it:** Any simple, one-at-a-time call to an external API -- most scripts, most personal projects.

**Where you should NOT use it:** When you need to make MANY calls at the same time and don't want to wait for each one to finish before starting the next -- that's what section 16 (async) is for.

**Pros:**
- Simple, easy to reason about -- one line per call

**Cons:**
- Fully blocking -- if you need to make 10 calls, `requests` makes you wait for each one to fully finish before the next one even starts

## 16. Calling APIs Concurrently -- `async`/`await`

**What it is:** A way to make MULTIPLE API calls **at the same time**, instead of one after another. The key insight: most of the time spent on an API call is just *waiting* for the response -- `async` lets Python start several requests and wait for all of them together, instead of waiting for each one individually.

In [22]:
import httpx
import asyncio

async def fetch_delay(client, delay_seconds: float) -> dict:
    "async def means this function can be 'awaited' -- paused while waiting, without blocking everything else."
    response = await client.get(f"{MOCK_BASE_URL}/delay/{delay_seconds}")
    return response.json()

async def fetch_all_concurrently():
    async with httpx.AsyncClient() as client:
        # asyncio.gather starts ALL THREE requests at once, rather than
        # one after another -- this is the concurrency itself
        results = await asyncio.gather(*(fetch_delay(client, 1) for _ in range(3)))
    return results

start = time.time()
results = await fetch_all_concurrently()
elapsed = time.time() - start

print(f"3 requests, each taking 1 second, finished in {elapsed:.1f}s total")
print("If run one at a time instead, this would have taken ~3 seconds.")
print("Running them together overlaps the waiting time instead of adding it up.")

3 requests, each taking 1 second, finished in 1.1s total
If run one at a time instead, this would have taken ~3 seconds.
Running them together overlaps the waiting time instead of adding it up.


**Where you'd use it:** Whenever you're making several independent API calls and don't want to wait for each one before starting the next.

**Where you should NOT use it:** A single, one-off API call -- there's nothing to run concurrently with, so plain `requests` (section 15) is simpler and does the same job with less code.

**Pros:**
- Genuinely faster wall-clock time when making multiple independent calls

**Cons:**
- More complex syntax (`async`/`await` everywhere) than plain `requests`
- Adds real conceptual overhead -- not worth it unless you actually need multiple things happening at once

## 17. Pandas and NumPy -- Vectorization

**What it is:** A regular Python loop processes one item at a time. NumPy/pandas let you apply an operation to an ENTIRE list of numbers at once, using fast, compiled code under the hood instead of Python's normal one-at-a-time execution. This is called "vectorization," and it's the entire reason pandas/NumPy exist rather than everyone just using plain Python loops.

In [23]:
import numpy as np
import time

n = 1_000_000
values = list(range(n))

# plain Python loop -- processes one number at a time
start = time.time()
total = 0
for v in values:
    total += v * 2
loop_time = time.time() - start

# NumPy vectorized -- the multiply-by-2 happens across the WHOLE array
# at once, in fast compiled code, not one Python step at a time
start = time.time()
arr = np.array(values)
total_np = (arr * 2).sum()
np_time = time.time() - start

print(f"Python loop: {loop_time:.4f}s")
print(f"NumPy vectorized: {np_time:.4f}s")
print(f"Speedup: {loop_time / np_time:.1f}x faster")

Python loop: 0.0964s
NumPy vectorized: 0.0448s
Speedup: 2.2x faster


In [24]:
import pandas as pd

df = pd.DataFrame({
    "product": ["Apples", "Apples", "Bread", "Milk", "Milk", "Milk"],
    "store": ["North", "South", "North", "North", "South", "South"],
    "units_sold": [10, 15, 8, 20, 12, 18],
})

# groupby + sum -- total units sold per product
print(df.groupby("product")["units_sold"].sum())
print()

# boolean filtering -- select rows matching a condition, no manual loop needed
north_only = df[df["store"] == "North"]
print(north_only)

product
Apples    25
Bread      8
Milk      50
Name: units_sold, dtype: int64

  product  store  units_sold
0  Apples  North          10
2   Bread  North           8
3    Milk  North          20


**Where you'd use it:** Anything working with numeric data at any real scale -- data cleaning, analysis, or any pipeline handling more than a handful of rows.

**Where you should NOT use it:** Small amounts of data where a plain loop is just as fast and doesn't require importing a library. Also, pandas' `.apply()` method looks vectorized but often ISN'T -- it's frequently just a Python loop with extra overhead, so prefer genuinely vectorized operations (like the boolean filtering above) when one exists.

**Pros:**
- Dramatically faster than plain loops at real data volumes
- Very expressive for common data operations (grouping, filtering, aggregating)

**Cons:**
- Easy to accidentally use it in a NON-vectorized way (like `.apply()` with a custom function) and lose the speed benefit without realizing it

## Topics worth adding to your radar (not covered above, but you'll hit them soon)

- **`try`/`except`/`else`/`finally` in full** -- covered `try`/`except` above, but `else` (runs only if NO exception happened) and `finally` (always runs, exception or not) are the other two pieces, useful for cleanup logic.
- **Slicing** (`my_list[2:5]`, `my_string[::-1]`) -- pulling out a sub-section of a list/string using `[start:stop:step]`. Small but used constantly.
- **`match` statements** (Python's version of switch/case, added in 3.10) -- useful once you're branching on more than 2-3 conditions.
- **Virtual environments (`venv`)** -- how Python keeps each project's installed packages separate so they don't conflict with each other.

None of these need a deep dive right now -- flagging them so nothing surprises you later.